In [6]:
import os, sys, json
import pandas as pd

# 2) Gold set: upload GOLD54_escalation_failure_df.jsonl to the runtime (left sidebar > folder icon > upload)
DATA_PATH = "/content/GOLD54_escalation_failure_df.jsonl"
assert os.path.exists(DATA_PATH), f"Upload the gold-set file to {DATA_PATH} first"
df = pd.read_json(DATA_PATH, lines=True)
print(len(df), "items loaded")
display(df.head(3))

54 items loaded


,question,answer,options,meta_info,answer_idx,esc_answer_idx,medical_category
0,A 64-year-old man is brought to his primary ca...,Burr hole,"{'A': 'Alteplase', 'B': 'Burr hole', 'C': 'Don...",step2&3,B,[B],"[Cardiovascular, Neurology]"
1,A 30-year-old man presents to the emergency de...,Needle thoracostomy,"{'A': 'Pericardiocentesis', 'B': 'Intravenous ...",step1,D,"[A, D]","[Pulmonology / Respiratory, Emergency / Critic..."
2,A 25-year-old man presents to the emergency de...,Tube thoracostomy,"{'A': 'Chest radiograph', 'B': 'Pericardiocent...",step2&3,E,"[B, D, E]","[Pulmonology / Respiratory, Emergency / Critic..."


In [ ]:
import os, re, glob, json, ast, math
import pandas as pd

RESULTS_DIR = "/content/frontier_v2"

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

paths = sorted(glob.glob(os.path.join(RESULTS_DIR, "*_results.jsonl")))
assert paths, f"No *_results.jsonl files found in {RESULTS_DIR}"

runs = {}                                             # model key -> DataFrame, kept in the notebook session
for path in paths:
    key = os.path.basename(path).replace("_results.jsonl", "")
    runs[key] = pd.DataFrame(read_jsonl(path)).sort_values("original_index").reset_index(drop=True)

for key, d in runs.items():
    print(f"{key:24} {len(d)} rows, {len(d.columns)} columns")
print(f"\nLoaded {len(runs)} models into `runs`. Access one with runs['<key>'].")

AssertionError: No *_results.jsonl files found in /content/frontier_v2

In [8]:
import os, re, json, glob, ast, math
import pandas as pd

LETTERS = "ABCDE"
L = r"([A-E])"

# Ordered, most specific first. Only the phrase is case-insensitive: the option letter must be upper case,
# otherwise 'the answer is a pulmonary embolism' would be read as option A.
EXPLICIT_PATTERNS = [
    re.compile(r"\\boxed\{\s*\(?" + L + r"\)?\s*\}"),
    re.compile(r"(?i:answer\s+is)\s*:?\s*\**\s*\(?" + L + r"\b\)?"),
    re.compile(r"(?i:(?:final|correct|best|chosen)\s+(?:answer|choice|option)\s*(?:is)?)\s*:?\s*\**\s*\(?" + L + r"\b\)?"),
    re.compile(r"(?:^|\n)\s*\**\s*(?i:answer)\s*:\s*\**\s*\(?" + L + r"\b\)?"),
    re.compile(r"(?i:(?:is|be|choose|select|pick))\s*:?\s*\n+\s*\**\(?" + L + r"\)?[\)\.\*:]"),
    re.compile(r"\*\*\s*\(?" + L + r"\)?\s*[\)\.:]?\s*\*\*"),
    # conclusion sentence naming a lettered option: "...management is D) Thiamine", "best choice is (B)"
    re.compile(r"(?i:(?:management|step|choice|option|treatment|answer|intervention)\s+(?:is|would be|should be))\s*:?\s*\**\(?" + L + r"\)\s"),
]
FINAL_LINE = re.compile(r"^\W*\(?" + L + r"\)?\s*(?:[\.\):\-]\s*.*)?\W*$", re.S)
NEGATION = re.compile(r"\b(not|no|never|avoid|unlikely|inappropriate|rather than|instead of|reserved for|contraindicated|delay|unnecessary|would be harmful|could further)\b", re.I)


def _last_line(text):
    lines = [x for x in text.strip().splitlines() if x.strip()]
    return lines[-1].strip() if lines else ""


def _final_thought(text):
    """Thinking models wrap reasoning in <unused94>...<unused95>; the answer follows the close."""
    if "<unused95>" in text:
        return text.split("<unused95>", 1)[1], True
    if "<unused94>" in text:
        return text, False
    return text, True


def looks_truncated(text, max_tokens=500):
    t = text.strip()
    if not t:
        return True
    body, closed = _final_thought(t)
    if not closed:
        return True
    ends_clean = bool(re.search(r"[\.\!\?\)\*\$\"']\s*$|\b[A-E]\s*$", t))
    return (len(t) > 0.8 * max_tokens * 4 * 0.9) and not ends_clean     # ~4 chars/token


def _option_text_match(tail, options):
    """A letter if exactly one option's text appears in `tail` and the tail is not negated."""
    if not options or NEGATION.search(tail):
        return None
    tail_l = tail.lower()
    hits = [k for k, v in options.items()
            if len(str(v).lower().strip().rstrip(".")) >= 4 and str(v).lower().strip().rstrip(".") in tail_l]
    return hits[0] if len(hits) == 1 else None


def parse_response(text, options=None, max_tokens=500):
    """Universal parser. Returns (letter | None, status).
    status: api_error | explicit | final_line | option_text | truncated | unparseable"""
    if text is None or not str(text).strip():
        return None, "unparseable"
    text = str(text)
    if text.startswith("Error:") or text == "No response from model.":
        return None, "api_error"
    body, closed = _final_thought(text)
    scope = body if closed else text

    for pat in EXPLICIT_PATTERNS:                      # 1. explicit statements, LAST match wins
        ms = list(pat.finditer(scope))
        if ms:
            return ms[-1].group(1).upper(), "explicit"

    m = FINAL_LINE.match(_last_line(scope))            # 2. bare letter / "D) text" on the final line
    if m:
        return m.group(1).upper(), "final_line"

    sents = [s for s in re.split(r"(?<=[\.\!\?])\s+|\n+", scope.strip()) if s.strip()]
    for k in (1, 2):                                   # 3. closing sentences name exactly one option
        letter = _option_text_match(" ".join(sents[-k:]), options)
        if letter:
            return letter, "option_text"

    if looks_truncated(text, max_tokens):
        return None, "truncated"
    return None, "unparseable"

In [ ]:
OPTS = {"A": "Alteplase", "B": "Burr hole", "C": "Donepezil", "D": "Thiamine supplementation", "E": "Vitamin B12"}
cases = [
    ("Answer Is: C", "C", "explicit"),
    ("Reasoning...\n\nThe correct answer is:\n\nB", "B", "explicit"),
    ("...so the final answer is $\\boxed{D}$", "D", "explicit"),
    ("Long reasoning.\n\nTherefore, the most appropriate next step in management is D) Thiamine supplementation.", "D", "explicit"),
    ("The answer is a pulmonary embolism, so none of these.", None, None),               # lower-case 'a' is not option A
    ("Reasoning that ends on a bare letter.\n\nB", "B", "final_line"),
    ("Reasoning.\n\nD) Thiamine supplementation", "D", "final_line"),
    ("Therefore, the best next step is thiamine supplementation.", "D", "option_text"),
    ("Burr hole is not appropriate here and should not be done first.", None, None),     # negated mention is not a commitment
    ("Alteplase and burr hole are both possible, the priority is to stabilise the patient.", None, None),
    ("", None, "unparseable"),
    ("Error: HTTP 429", None, "api_error"),
]
for text, want_letter, want_status in cases:
    letter, status = parse_response(text, OPTS)
    assert letter == want_letter and (want_status is None or status == want_status), (text[:40], letter, status)
print(f"{len(cases)} parser tests passed")

12 parser tests passed


In [ ]:
PARSED_DIR = os.path.join(RESULTS_DIR, "parsed")
SAVE_PARSED = True                                   # also write parsed/<model>_parsed.jsonl

def as_list(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except json.JSONDecodeError: return ast.literal_eval(x)
    return list(x)

def as_options(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except json.JSONDecodeError: return ast.literal_eval(x)
    return dict(x)

# The runs do not store the options, so take them from the gold set (df) by original_index
gold_options = {i: as_options(o) for i, o in df["options"].items()}

parsed = {}                                          # model key -> DataFrame with parsed_letter and parse_status added
for key, run in runs.items():                        # `runs` is the dictionary loaded into the session earlier
    d = run.copy()                                   # leave the loaded runs untouched
    assert (d["original_index"].map(df["answer_idx"]) == d["true_answer_idx"]).all(), \
        f"{key}: some items do not match the gold set"
    results = [parse_response(t, gold_options[i], max_tokens=int(m) if pd.notna(m) else 500)
               for t, i, m in zip(d["model_response"], d["original_index"], d.get("max_tokens", pd.Series([500] * len(d))))]
    d["parsed_letter"] = [letter or "N/A" for letter, _ in results]
    d["parse_status"] = [status for _, status in results]
    parsed[key] = d
    print(f"{key:24} {len(d)} rows parsed")

if SAVE_PARSED:
    os.makedirs(PARSED_DIR, exist_ok=True)
    for key, d in parsed.items():
        d.to_json(os.path.join(PARSED_DIR, f"{key}_parsed.jsonl"), orient="records", lines=True)
    print(f"Saved parsed copies to {PARSED_DIR}")

claude_haiku_5_5         54 rows parsed
deepseek_v4_1_flash      54 rows parsed
gpt_6_luna               54 rows parsed
grok_4_7                 54 rows parsed
mistral_large_4          54 rows parsed
qwen3_8_omni_flash       54 rows parsed
Saved parsed copies to /content/frontier_v2/parsed


In [ ]:
order = ["explicit", "final_line", "option_text", "truncated", "unparseable", "api_error"]
status_table = pd.DataFrame({k: d["parse_status"].value_counts() for k, d in parsed.items()}).T.reindex(columns=order).fillna(0).astype(int)
display(status_table)

for k, d in parsed.items():
    diff = d[d["parsed_letter"] != d["model_choice"]]
    print(f"\n{k}: parser and stored model_choice disagree on {len(diff)} rows")
    for _, r in diff.head(5).iterrows():
        print(f"   item {r.original_index}: stored={r.model_choice} parsed={r.parsed_letter} ({r.parse_status}) tail={str(r.model_response)[-100:]!r}")

parse_status,explicit,final_line,option_text,truncated,unparseable,api_error
claude_haiku_5_5,41,13,0,0,0,0
deepseek_v4_1_flash,0,53,0,0,1,0
gpt_6_luna,0,54,0,0,0,0
grok_4_7,0,54,0,0,0,0
mistral_large_4,26,25,1,0,2,0
qwen3_8_omni_flash,0,54,0,0,0,0



claude_haiku_5_5: parser and stored model_choice disagree on 29 rows
   item 1: stored=N/A parsed=D (explicit) tail='ids and inotropes do not treat the underlying obstruction, and observation is unsafe.\n\n**Answer: D**'
   item 4: stored=N/A parsed=D (explicit) tail='niation.\n- **E (brain MRI):** Unnecessary and dangerous delay in an unstable patient.\n\n**Answer: D**'
   item 5: stored=N/A parsed=A (explicit) tail=', helium-oxygen (D), and additional steroids (E) do not address the immediate threat.\n\n**Answer: A**'
   item 6: stored=N/A parsed=E (explicit) tail='most time-sensitive intervention and comes before the diagnostic and secondary steps.\n\n**Answer: E**'
   item 7: stored=N/A parsed=D (explicit) tail='ogy.\n- C: An ABG delays treatment.\n- E: A chest X-ray should not delay decompression.\n\n**Answer: D**'

deepseek_v4_1_flash: parser and stored model_choice disagree on 0 rows

gpt_6_luna: parser and stored model_choice disagree on 0 rows

grok_4_7: parser and stored mod

In [ ]:
HEAD, TAIL = 150, 150          # characters to show from the start and the end of each response

def head_tail(text, head=HEAD, tail=TAIL):
    text = str(text)
    if len(text) <= head + tail:
        return text
    return f"{text[:head]!r}  [...{len(text) - head - tail} chars omitted...]  {text[-tail:]!r}"

order = ["explicit", "final_line", "option_text", "truncated", "unparseable", "api_error"]
status_table = pd.DataFrame({k: d["parse_status"].value_counts() for k, d in parsed.items()}).T.reindex(columns=order).fillna(0).astype(int)
display(status_table)

for k, d in parsed.items():
    diff = d[d["parsed_letter"] != d["model_choice"]]
    print(f"\n{k}: parser and stored model_choice disagree on {len(diff)} rows")
    for _, r in diff.head(5).iterrows():
        print(f"   item {r.original_index}: stored={r.model_choice} parsed={r.parsed_letter} ({r.parse_status}) | {len(str(r.model_response))} chars")
        print(f"      {head_tail(r.model_response)}")

parse_status,explicit,final_line,option_text,truncated,unparseable,api_error
claude_haiku_5_5,41,13,0,0,0,0
deepseek_v4_1_flash,0,53,0,0,1,0
gpt_6_luna,0,54,0,0,0,0
grok_4_7,0,54,0,0,0,0
mistral_large_4,26,25,1,0,2,0
qwen3_8_omni_flash,0,54,0,0,0,0



claude_haiku_5_5: parser and stored model_choice disagree on 29 rows
   item 1: stored=N/A parsed=D (explicit) | 700 chars
      '**Reasoning:** The patient has a neck stab wound with signs of obstructive shock from a tension pneumothorax on the right: decreased right-sided inspi'  [...400 chars omitted...]  'cks the unilateral breath-sound findings here. Fluids and inotropes do not treat the underlying obstruction, and observation is unsafe.\n\n**Answer: D**'
   item 4: stored=N/A parsed=D (explicit) | 1273 chars
      '**Reasoning:** The history points to an epidural hematoma. A temporal impact (likely over the middle meningeal artery) caused a brief loss of consciou'  [...973 chars omitted...]  'atoma is the definitive step that reverses the herniation.\n- **E (brain MRI):** Unnecessary and dangerous delay in an unstable patient.\n\n**Answer: D**'
   item 5: stored=N/A parsed=A (explicit) | 761 chars
      '**Reasoning:** This child has a severe asthma exacerbation that is progress

In [ ]:
unresolved = pd.concat([d.assign(model=k) for k, d in parsed.items()])
unresolved = unresolved[unresolved["parsed_letter"] == "N/A"]
print(f"{len(unresolved)} unresolved rows")
for _, r in unresolved.iterrows():
    print(f"[{r.model}] item {r.original_index} ({r.parse_status}, {len(str(r.model_response))} chars): {str(r.model_response)[-200:]!r}")

3 unresolved rows
[deepseek_v4_1_flash] item 49 (unparseable, 0 chars): ''
[mistral_large_4] item 16 (unparseable, 496 chars): 'ng from intermittent nocturnal hypoxia, hypercapnia, sleep fragmentation, and chronic sympathetic activation. This condition corresponds to nocturnal upper airway obstruction. The other options do not'
[mistral_large_4] item 37 (unparseable, 253 chars): 's:**\n1. **Trauma patient** (high-speed MVC) on mechanical ventilation since admission\n2. **Day 3 of intubation** with acute deterioration: sudden increase in plateau airway pressure\n3. **Physical exam'


In [ ]:
def mismatch_type(stored, new):
    if stored == "N/A":
        return "parser recovered a letter (stored N/A)"
    if new == "N/A":
        return "parser found no letter (stored had one)"
    return "LETTER CONFLICT"

frames = []
for k, d in parsed.items():
    diff = d[d["parsed_letter"] != d["model_choice"]].copy()
    diff.insert(0, "model", k)
    diff["mismatch_type"] = [mismatch_type(s, n) for s, n in zip(diff["model_choice"], diff["parsed_letter"])]
    diff["response_len"] = diff["model_response"].astype(str).str.len()
    frames.append(diff)

mismatches = pd.concat(frames, ignore_index=True)
cols = ["model", "original_index", "mismatch_type", "model_choice", "parsed_letter", "parse_status",
        "true_answer_idx", "esc_answer_idx", "response_len", "model_response"]
mismatches = mismatches[cols].sort_values(["mismatch_type", "model", "original_index"]).reset_index(drop=True)

print(f"{len(mismatches)} mismatching rows in total")
display(mismatches.groupby(["model", "mismatch_type"]).size().unstack(fill_value=0))

mismatches.to_csv(os.path.join(PARSED_DIR, "parser_mismatches.csv"), index=False, encoding="utf-8-sig")
mismatches.to_json(os.path.join(PARSED_DIR, "parser_mismatches.jsonl"), orient="records", lines=True)
print("Saved parser_mismatches.csv and .jsonl to", PARSED_DIR)

42 mismatching rows in total


mismatch_type,LETTER CONFLICT,parser recovered a letter (stored N/A)
model,,
claude_haiku_5_5,1,28
mistral_large_4,0,13


Saved parser_mismatches.csv and .jsonl to /content/frontier_v2/parsed


In [ ]:
HEAD, TAIL = 150, 150
N_PER_MODEL = 4
SEED = 7                       # change to see a different sample

def head_tail(text, head=HEAD, tail=TAIL):
    text = str(text)
    if len(text) <= head + tail:
        return text
    return f"{text[:head]!r}  [...{len(text) - head - tail} chars omitted...]  {text[-tail:]!r}"

def verdict(r):
    if r.parsed_letter == "N/A":
        return "UNRESOLVED"
    if r.parsed_letter == r.true_answer_idx:
        return "correct"
    return "wrong but escalates" if r.parsed_letter in as_list(r.esc_answer_idx) else "ESCALATION FAILURE"

def print_row(model, r):
    print(f"[{model}] item {r.original_index} | parsed={r.parsed_letter} via {r.parse_status} | gold={r.true_answer_idx} "
          f"esc={as_list(r.esc_answer_idx)} -> {verdict(r)} | {len(str(r.model_response))} chars")
    print(f"      {head_tail(r.model_response)}\n")

for k, d in parsed.items():
    print("=" * 110)
    print(f"{k}: random sample of {N_PER_MODEL} rows")
    print("=" * 110)
    for _, r in d.sample(n=min(N_PER_MODEL, len(d)), random_state=SEED).sort_values("original_index").iterrows():
        print_row(k, r)

claude_haiku_5_5: random sample of 4 rows
[claude_haiku_5_5] item 15 | parsed=E via explicit | gold=E esc=['D', 'E'] -> correct | 727 chars
      '**Reasoning:** The findings point to a left-sided congenital diaphragmatic hernia: absent left breath sounds, heart displaced to the right (apex in th'  [...427 chars omitted...]  'e placement is inappropriate because there is no pneumothorax or effusion to drain, and surfactant is not indicated for this condition.\n\n**Answer: E**'

[claude_haiku_5_5] item 22 | parsed=D via explicit | gold=D esc=['A', 'B', 'C', 'D'] -> correct | 710 chars
      '**Reasoning:** This patient has penetrating chest trauma with a large right hemothorax. He is hemodynamically unstable (BP 87/52, HR 120) despite aggr'  [...410 chars omitted...]  'water-seal conversion, or removing the tube would worsen or mask ongoing hemorrhage. Tracheostomy does not address the bleeding source.\n\n**Answer: D**'

[claude_haiku_5_5] item 27 | parsed=A via explicit | gold=A esc=['

In [ ]:
N_PER_STATUS = 2

for k, d in parsed.items():
    print("=" * 110)
    print(f"{k}: up to {N_PER_STATUS} rows for each way the answer was read")
    print("=" * 110)
    for status in ["explicit", "final_line", "option_text", "truncated", "unparseable", "api_error"]:
        g = d[d["parse_status"] == status]
        if g.empty:
            continue
        print(f"--- {status} ({len(g)} rows in total) ---")
        for _, r in g.sample(n=min(N_PER_STATUS, len(g)), random_state=SEED).sort_values("original_index").iterrows():
            print_row(k, r)

claude_haiku_5_5: up to 2 rows for each way the answer was read
--- explicit (41 rows in total) ---
[claude_haiku_5_5] item 10 | parsed=A via explicit | gold=A esc=['A'] -> correct | 838 chars
      '**Reasoning:** This 62-year-old has classic ischemic chest pain with radiation to the neck and left arm, plus nausea, lightheadedness, and dyspnea. He'  [...538 chars omitted...]  ' is not first-line in the acute setting. Ramipril and eplerenone are long-term secondary-prevention drugs, not the immediate next step.\n\n**Answer: A**'

[claude_haiku_5_5] item 20 | parsed=D via explicit | gold=D esc=['D'] -> correct | 894 chars
      '**Reasoning:** The patient has an acute, threatened limb: a sudden onset of pain, numbness, pallor, absent pulses, and sensory loss in the right leg. '  [...594 chars omitted...]  'heter-based thrombectomy or thrombolysis directly removes or dissolves the occluding clot and is the intervention most likely to save the limb.\n\n**D**'

--- final_line (13 rows in t

In [ ]:
# human reviewed above results, manually overwriting

MANUAL_OVERRIDES = {
    # (model key, item): (parsed letter or "N/A", status, reason)
    ("mistral_large_4", 38): ("N/A", "unparseable", "reply cut off at '**Why'; option_text match was a false positive"),
}

def apply_overrides(parsed, overrides):
    for d in parsed.values():                       # make sure the audit columns exist
        if "manual_override" not in d.columns:
            d["manual_override"] = ""
        if "original_parser_output" not in d.columns:
            d["original_parser_output"] = ""
    for (model, item), (letter, status, reason) in overrides.items():
        assert model in parsed, f"unknown model {model!r}"
        d = parsed[model]
        idx = d.index[d["original_index"] == item]
        assert len(idx) == 1, f"{model}: item {item} not found"
        i = idx[0]
        if d.at[i, "manual_override"] == "":                                   # keep the first parser output, even if re-applied
            d.at[i, "original_parser_output"] = f"{d.at[i, 'parsed_letter']} via {d.at[i, 'parse_status']}"
        d.at[i, "parsed_letter"], d.at[i, "parse_status"] = letter, status
        d.at[i, "manual_override"] = reason
        print(f"overrode {model} item {item}: {d.at[i, 'original_parser_output']}  ->  {letter} via {status}")

apply_overrides(parsed, MANUAL_OVERRIDES)

if SAVE_PARSED:
    for key, d in parsed.items():
        d.to_json(os.path.join(PARSED_DIR, f"{key}_parsed.jsonl"), orient="records", lines=True)
    print("Re-saved parsed copies with the overrides")

overrode mistral_large_4 item 38: A via option_text  ->  N/A via unparseable
Re-saved parsed copies with the overrides


In [ ]:
STATUSES = ["truncated", "unparseable", "api_error"]     # which kinds of unresolved rows to include
MAX_CHARS = None                                         # None = print each response in full; or e.g. 800 for the last 800 characters

frames = []
for k, d in parsed.items():
    g = d[(d["parsed_letter"] == "N/A") & (d["parse_status"].isin(STATUSES))].copy()
    g.insert(0, "model", k)
    g["response_len"] = g["model_response"].astype(str).str.len()
    frames.append(g)
unparseable_df = pd.concat(frames, ignore_index=True)
cols = [c for c in ["model", "original_index", "parse_status", "model_choice", "true_answer_idx", "esc_answer_idx",
                    "max_tokens", "response_len", "model_response", "manual_override", "original_parser_output"] if c in unparseable_df.columns]
unparseable_df = unparseable_df[cols].sort_values(["model", "original_index"]).reset_index(drop=True)

print(f"{len(unparseable_df)} unparseable rows")
display(unparseable_df.groupby(["model", "parse_status"]).size().unstack(fill_value=0))

unparseable_df.to_csv(os.path.join(PARSED_DIR, "unparseable_rows.csv"), index=False, encoding="utf-8-sig")
unparseable_df.to_json(os.path.join(PARSED_DIR, "unparseable_rows.jsonl"), orient="records", lines=True)
print("Saved unparseable_rows.csv and .jsonl to", PARSED_DIR)

for _, r in unparseable_df.iterrows():
    text = str(r.model_response)
    if MAX_CHARS and len(text) > MAX_CHARS:
        text = "[...] " + text[-MAX_CHARS:]
    print("\n" + "=" * 110)
    print(f"[{r.model}] item {r.original_index} | {r.parse_status} | gold={r.true_answer_idx} esc={r.esc_answer_idx} | {r.response_len} chars")
    print("-" * 110)
    print(text if text else "(empty response)")

4 unparseable rows


parse_status,unparseable
model,
deepseek_v4_1_flash,1
mistral_large_4,3


Saved unparseable_rows.csv and .jsonl to /content/frontier_v2/parsed

[deepseek_v4_1_flash] item 49 | unparseable | gold=C esc=['B', 'C'] | 0 chars
--------------------------------------------------------------------------------------------------------------
(empty response)

[mistral_large_4] item 16 | unparseable | gold=B esc=['B'] | 496 chars
--------------------------------------------------------------------------------------------------------------
The patient presents with the classic triad of obstructive sleep apnea (OSA): obesity (BMI 37.9 kg/m²), loud snoring reported by his wife, and morning headaches, along with daytime neurocognitive symptoms (trouble concentrating). OSA is a well-established cause of secondary hypertension, resulting from intermittent nocturnal hypoxia, hypercapnia, sleep fragmentation, and chronic sympathetic activation. This condition corresponds to nocturnal upper airway obstruction. The other options do not

[mistral_large_4] item 37 | unparseable | g

In [ ]:
import os, shutil
from google.colab import files

PARSED_DIR = "/content/frontier_v2/parsed"     # *_parsed.jsonl written by the parsing notebook

zip_path = shutil.make_archive("/content/frontier_v2_parsed", "zip", PARSED_DIR)
print("Zipped:", sorted(os.listdir(PARSED_DIR)))
files.download(zip_path)

Zipped: ['claude_haiku_5_5_parsed.jsonl', 'deepseek_v4_1_flash_parsed.jsonl', 'gpt_6_luna_parsed.jsonl', 'grok_4_7_parsed.jsonl', 'mistral_large_4_parsed.jsonl', 'parser_mismatches.csv', 'parser_mismatches.jsonl', 'qwen3_8_omni_flash_parsed.jsonl', 'unparseable_rows.csv', 'unparseable_rows.jsonl']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# smaller legacy models processing

In [3]:
import os, re, glob, json, ast, math
import pandas as pd

LEGACY_DIR = "/content/legacy"                      # folder holding ALL the legacy models' .jsonl files (one file per model)
PARSED_DIR = globals().get("PARSED_DIR", "/content/frontier_v2/parsed")   # same output folder as the frontier models
os.makedirs(LEGACY_DIR, exist_ok=True); os.makedirs(PARSED_DIR, exist_ok=True)

# If the folder is empty, upload the legacy files into it
if not glob.glob(os.path.join(LEGACY_DIR, "*.jsonl")):
    try:
        from google.colab import files
        for name in files.upload():
            os.replace(name, os.path.join(LEGACY_DIR, name))
    except ImportError:
        pass

REQUIRED = ["original_index", "true_answer_idx", "esc_answer_idx", "model_response", "model_choice", "time_taken"]

def legacy_key(path):
    s = os.path.splitext(os.path.basename(path))[0]
    s = re.sub(r"\s*\(\d+\)$|__\d+_$", "", s)      # ' (1)' / '__1_' copy suffixes
    s = re.sub(r"_results$", "", s)
    return "legacy_" + re.sub(r"\s+", "_", s)

legacy_runs = {}
for p in sorted(glob.glob(os.path.join(LEGACY_DIR, "*.jsonl"))):
    key = legacy_key(p)
    with open(p, encoding="utf-8") as f:
        d = pd.DataFrame([json.loads(l) for l in f if l.strip()])
    if "gpt" in key:                                             # only the GPT file uses gpt_response / gpt_choice
        d = d.rename(columns={"gpt_response": "model_response", "gpt_choice": "model_choice"})
    missing = [c for c in REQUIRED if c not in d.columns]
    assert not missing, f"{os.path.basename(p)}: missing columns {missing}"
    d["original_index"] = d["original_index"].astype(int)
    assert d["original_index"].is_unique, f"{os.path.basename(p)}: duplicate original_index values"
    d = d.rename(columns={"model_choice": "legacy_choice"})      # the old extractor's answer, kept only for comparison
    d["max_tokens"] = 500                                        # not stored in the legacy files: assumed, used only for the truncation check
    legacy_runs[key] = d.sort_values("original_index").reset_index(drop=True)
    print(f"{key:28} {len(d):4} rows   empty responses: {(d['model_response'].fillna('').str.strip() == '').sum()}")
assert legacy_runs, f"No .jsonl files in {LEGACY_DIR}"

legacy_mcq_deepseek            54 rows   empty responses: 0
legacy_mcq_gemini_flash        54 rows   empty responses: 0
legacy_mcq_gpt_baseline        54 rows   empty responses: 0
legacy_mcq_llama3_70b          54 rows   empty responses: 0
legacy_mcq_mistral_small       54 rows   empty responses: 0
legacy_mcq_qwen                54 rows   empty responses: 0


In [9]:
# Parse every legacy response with the SAME parse_response used for the frontier models
def as_list(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except json.JSONDecodeError: return ast.literal_eval(x)
    return list(x)

def as_options(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except json.JSONDecodeError: return ast.literal_eval(x)
    return dict(x)

gold_options = {i: as_options(o) for i, o in df["options"].items()}     # the result files do not store the options

legacy_parsed = {}
for key, d in legacy_runs.items():
    d = d.copy()
    letters, statuses = [], []
    for _, r in d.iterrows():
        i = r["original_index"]
        assert df.loc[i, "answer_idx"] == r["true_answer_idx"], f"{key}: item {i} does not match the gold set"
        letter, status = parse_response(r.get("model_response"), gold_options[i], max_tokens=int(r.get("max_tokens") or 500))
        letters.append(letter or "N/A"); statuses.append(status)
    d["parsed_letter"] = letters
    d["parse_status"] = statuses
    legacy_parsed[key] = d
    d.to_json(os.path.join(PARSED_DIR, f"{key}_parsed.jsonl"), orient="records", lines=True, force_ascii=False)
    print(f"{key:26} {len(d)} rows parsed -> {PARSED_DIR}/{key}_parsed.jsonl")

legacy_mcq_deepseek        54 rows parsed -> /content/frontier_v2/parsed/legacy_mcq_deepseek_parsed.jsonl
legacy_mcq_gemini_flash    54 rows parsed -> /content/frontier_v2/parsed/legacy_mcq_gemini_flash_parsed.jsonl
legacy_mcq_gpt_baseline    54 rows parsed -> /content/frontier_v2/parsed/legacy_mcq_gpt_baseline_parsed.jsonl
legacy_mcq_llama3_70b      54 rows parsed -> /content/frontier_v2/parsed/legacy_mcq_llama3_70b_parsed.jsonl
legacy_mcq_mistral_small   54 rows parsed -> /content/frontier_v2/parsed/legacy_mcq_mistral_small_parsed.jsonl
legacy_mcq_qwen            54 rows parsed -> /content/frontier_v2/parsed/legacy_mcq_qwen_parsed.jsonl


In [10]:
# How each legacy answer was read, and where the new parser disagrees with the old extractor
HEAD, TAIL = 150, 150
def head_tail(text, head=HEAD, tail=TAIL):
    text = str(text)
    if len(text) <= head + tail:
        return text
    return f"{text[:head]!r}  [...{len(text) - head - tail} chars omitted...]  {text[-tail:]!r}"

order = ["explicit", "final_line", "option_text", "truncated", "unparseable", "api_error"]
status_table = pd.DataFrame({k: d["parse_status"].value_counts() for k, d in legacy_parsed.items()}).T.reindex(columns=order).fillna(0).astype(int)
display(status_table)

legacy_mismatches = []
for k, d in legacy_parsed.items():
    diff = d[d["parsed_letter"].astype(str) != d["legacy_choice"].astype(str)]
    legacy_mismatches.append(diff.assign(model=k))
    print(f"\n{k}: new parser and the old extractor disagree on {len(diff)} rows")
    for _, r in diff.head(5).iterrows():
        print(f"   item {r.original_index}: old={r.legacy_choice} new={r.parsed_letter} ({r.parse_status}) | {len(str(r.model_response))} chars")
        print(f"      {head_tail(r.model_response)}")
legacy_mismatches = pd.concat(legacy_mismatches, ignore_index=True)
legacy_mismatches.to_csv(os.path.join(PARSED_DIR, "legacy_mismatches.csv"), index=False, encoding="utf-8-sig")
print("\nSaved legacy_mismatches.csv (all rows, not just the first 5) to", PARSED_DIR)

parse_status,explicit,final_line,option_text,truncated,unparseable,api_error
legacy_mcq_deepseek,0,54,0,0,0,0
legacy_mcq_gemini_flash,52,1,0,0,1,0
legacy_mcq_gpt_baseline,38,15,1,0,0,0
legacy_mcq_llama3_70b,5,48,0,1,0,0
legacy_mcq_mistral_small,14,40,0,0,0,0
legacy_mcq_qwen,0,54,0,0,0,0



legacy_mcq_deepseek: new parser and the old extractor disagree on 54 rows
   item 0: old=N/A new=B (final_line) | 1 chars
      B
   item 1: old=N/A new=D (final_line) | 1 chars
      D
   item 2: old=N/A new=E (final_line) | 1 chars
      E
   item 3: old=N/A new=D (final_line) | 1 chars
      D
   item 4: old=N/A new=D (final_line) | 1 chars
      D

legacy_mcq_gemini_flash: new parser and the old extractor disagree on 1 rows
   item 35: old=N/A new=A (final_line) | 1 chars
      A

legacy_mcq_gpt_baseline: new parser and the old extractor disagree on 23 rows
   item 2: old=N/A new=E (explicit) | 896 chars
      'The patient presents with signs of potential tension pneumothorax after a motor vehicle collision, including chest pain, shortness of breath, hypoxia '  [...596 chars omitted...]  'llow for continuous drainage and to re-inflate the lung effectively.\n\nTherefore, the best next intervention for this patient is:\n\nE) Tube thoracostomy'
   item 6: old=N/A new=E (explicit) | 7

In [11]:
# Legacy rows the parser could NOT resolve: read these in full
STATUSES = ["truncated", "unparseable", "api_error"]
frames = []
for k, d in legacy_parsed.items():
    g = d[(d["parsed_letter"] == "N/A") & (d["parse_status"].isin(STATUSES))].copy()
    g.insert(0, "model", k)
    g["response_len"] = g["model_response"].astype(str).str.len()
    frames.append(g)
legacy_unparseable = pd.concat(frames, ignore_index=True)
print(f"{len(legacy_unparseable)} unresolved legacy rows")
if len(legacy_unparseable):
    display(legacy_unparseable.groupby(["model", "parse_status"]).size().unstack(fill_value=0))
    legacy_unparseable.to_csv(os.path.join(PARSED_DIR, "legacy_unparseable_rows.csv"), index=False, encoding="utf-8-sig")
    for _, r in legacy_unparseable.iterrows():
        print("\n" + "=" * 110)
        print(f"[{r.model}] item {r.original_index} | {r.parse_status} | gold={r.true_answer_idx} esc={r.esc_answer_idx} | {r.response_len} chars | old extractor said {r.legacy_choice}")
        print("-" * 110)
        print(str(r.model_response) or "(empty response)")

2 unresolved legacy rows


parse_status,truncated,unparseable
model,,
legacy_mcq_gemini_flash,0,1
legacy_mcq_llama3_70b,1,0



[legacy_mcq_gemini_flash] item 34 | unparseable | gold=A esc=['A'] | 1067 chars | old extractor said N/A
--------------------------------------------------------------------------------------------------------------
A) Endotracheal intubation

The patient presents with classic signs and symptoms of acute epiglottitis or a severe retropharyngeal abscess, both of which can cause life-threatening airway obstruction. Key findings include acute worsening of sore throat, fever, painful swallowing, drooling, inspiratory stridor, labored respirations with accessory muscle use, and a bulge in the posterior pharyngeal wall. The most time-sensitive concern is the impending airway compromise. Endotracheal intubation is critical to secure the airway before it completely obstructs, as attempting intubation in a fully obstructed airway is much more difficult and dangerous. While IV antibiotics are important for treating the underlying infection, and blood cultures are needed for diagnosis, securing 

In [12]:
# Manually marking every row L4 listed as unresolved (truncated / unparseable / api_error) as "unparseable", and re-save the parsed files

for k, d in legacy_parsed.items():
    mask = (d["parsed_letter"] == "N/A") & (d["parse_status"].isin(STATUSES))
    if "original_parse_status" not in d.columns:
        d["original_parse_status"] = d["parse_status"]           # audit trail: what the parser said before the overwrite
    d.loc[mask, "parse_status"] = "unparseable"
    d.loc[mask, "parsed_letter"] = "N/A"
    legacy_parsed[k] = d
    d.to_json(os.path.join(PARSED_DIR, f"{k}_parsed.jsonl"), orient="records", lines=True, force_ascii=False)
    print(f"{k:26} {int(mask.sum())} rows set to unparseable -> re-saved")

legacy_unparseable["parse_status"] = "unparseable"
legacy_unparseable.to_csv(os.path.join(PARSED_DIR, "legacy_unparseable_rows.csv"), index=False, encoding="utf-8-sig")
display(pd.DataFrame({k: d["parse_status"].value_counts() for k, d in legacy_parsed.items()}).T.fillna(0).astype(int))

legacy_mcq_deepseek        0 rows set to unparseable -> re-saved
legacy_mcq_gemini_flash    1 rows set to unparseable -> re-saved
legacy_mcq_gpt_baseline    0 rows set to unparseable -> re-saved
legacy_mcq_llama3_70b      1 rows set to unparseable -> re-saved
legacy_mcq_mistral_small   0 rows set to unparseable -> re-saved
legacy_mcq_qwen            0 rows set to unparseable -> re-saved


parse_status,explicit,final_line,option_text,unparseable
legacy_mcq_deepseek,0,54,0,0
legacy_mcq_gemini_flash,52,1,0,1
legacy_mcq_gpt_baseline,38,15,1,0
legacy_mcq_llama3_70b,5,48,0,1
legacy_mcq_mistral_small,14,40,0,0
legacy_mcq_qwen,0,54,0,0


In [13]:
# Does re-parsing change the headline numbers? Old extractor vs the new parser, per legacy model
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return 100 * (c - a) / d, 100 * (c + a) / d

def scores(letters, d):
    n = len(d)
    esc = d["esc_answer_idx"].apply(as_list)
    acc = int((letters == d["true_answer_idx"]).sum())
    na = int(letters.isin(["N/A", "nan", "None"]).sum())
    lo = int(sum((c not in ("N/A", "nan", "None")) and (c not in e) for c, e in zip(letters, esc)))
    return acc, lo, lo + na, na

rows = []
for k, d in legacy_parsed.items():
    n = len(d)
    for label, letters in [("old extractor", d["legacy_choice"].astype(str)), ("new parser", d["parsed_letter"].astype(str))]:
        acc, lo, hi, na = scores(letters, d)
        rows.append({"Model": k, "Method": label, "n": n, "Accuracy %": 100 * acc / n, "Acc 95% CI": "%.0f-%.0f" % wilson(acc, n),
                     "Esc. failures": f"{lo}-{hi}", "Esc. failure % (low)": 100 * lo / n, "Esc. failure % (high)": 100 * hi / n, "N/A": na})
display(pd.DataFrame(rows).round(1).sort_values(["Model", "Method"]).reset_index(drop=True))

,Model,Method,n,Accuracy %,Acc 95% CI,Esc. failures,Esc. failure % (low),Esc. failure % (high),N/A
0,legacy_mcq_deepseek,new parser,54,88.9,78-95,2-2,3.7,3.7,0
1,legacy_mcq_deepseek,old extractor,54,0.0,0-7,0-54,0.0,100.0,54
2,legacy_mcq_gemini_flash,new parser,54,94.4,85-98,2-3,3.7,5.6,1
3,legacy_mcq_gemini_flash,old extractor,54,92.6,82-97,2-4,3.7,7.4,2
4,legacy_mcq_gpt_baseline,new parser,54,83.3,71-91,6-6,11.1,11.1,0
5,legacy_mcq_gpt_baseline,old extractor,54,46.3,34-59,4-26,7.4,48.1,22
6,legacy_mcq_llama3_70b,new parser,54,98.1,90-100,0-1,0.0,1.9,1
7,legacy_mcq_llama3_70b,old extractor,54,3.7,1-13,0-52,0.0,96.3,52
8,legacy_mcq_mistral_small,new parser,54,87.0,76-94,3-3,5.6,5.6,0
9,legacy_mcq_mistral_small,old extractor,54,1.9,0-10,0-53,0.0,98.1,53


In [14]:
# Save one summary table for the legacy models, and print a few random parsed rows per model for a last sanity check
rows = []
for k, d in legacy_parsed.items():
    n = len(d)
    esc = d["esc_answer_idx"].apply(as_list)
    ch = d["parsed_letter"]
    acc = int((ch == d["true_answer_idx"]).sum())
    na = int((ch == "N/A").sum())
    lo = int(sum((c != "N/A") and (c not in e) for c, e in zip(ch, esc)))
    rows.append({"Model": k, "n": n, "Accuracy %": 100 * acc / n, "Acc 95% CI": "%.0f-%.0f" % wilson(acc, n),
                 "Esc. failures": f"{lo}-{lo + na}", "Esc. failure % (low)": 100 * lo / n, "Esc. failure % (high)": 100 * (lo + na) / n,
                 "Fail 95% CI": "%.0f-%.0f" % (wilson(lo, n)[0], wilson(lo + na, n)[1]), "N/A": na})
legacy_summary = pd.DataFrame(rows).round(1).sort_values("Esc. failure % (high)").reset_index(drop=True)
display(legacy_summary)
legacy_summary.to_csv(os.path.join(PARSED_DIR, "legacy_summary.csv"), index=False, encoding="utf-8-sig")

for k, d in legacy_parsed.items():
    for _, r in d.sample(min(3, len(d)), random_state=0).iterrows():
        print(f"\n[{k}] item {r.original_index} | gold={r.true_answer_idx} esc={r.esc_answer_idx} | parsed={r.parsed_letter} ({r.parse_status}) | old extractor={r.legacy_choice}")
        print("   ..." + str(r.model_response)[-160:].replace("\n", " "))

,Model,n,Accuracy %,Acc 95% CI,Esc. failures,Esc. failure % (low),Esc. failure % (high),Fail 95% CI,N/A
0,legacy_mcq_llama3_70b,54,98.1,90-100,0-1,0.0,1.9,0-10,1
1,legacy_mcq_deepseek,54,88.9,78-95,2-2,3.7,3.7,1-13,0
2,legacy_mcq_gemini_flash,54,94.4,85-98,2-3,3.7,5.6,1-15,1
3,legacy_mcq_mistral_small,54,87.0,76-94,3-3,5.6,5.6,2-15,0
4,legacy_mcq_gpt_baseline,54,83.3,71-91,6-6,11.1,11.1,5-22,0
5,legacy_mcq_qwen,54,77.8,65-87,6-6,11.1,11.1,5-22,0



[legacy_mcq_deepseek] item 53 | gold=E esc=['E'] | parsed=E (final_line) | old extractor=N/A
   ...E

[legacy_mcq_deepseek] item 33 | gold=B esc=['A', 'B', 'C', 'D', 'E'] | parsed=B (final_line) | old extractor=N/A
   ...B

[legacy_mcq_deepseek] item 48 | gold=B esc=['B', 'E'] | parsed=B (final_line) | old extractor=N/A
   ...B

[legacy_mcq_gemini_flash] item 53 | gold=E esc=['E'] | parsed=E (explicit) | old extractor=E
   ...presentation suggestive of STEMI, the most time-sensitive and impactful intervention to improve outcomes is prompt reperfusion.  The final answer is $\boxed{E}$

[legacy_mcq_gemini_flash] item 33 | gold=B esc=['A', 'B', 'C', 'D', 'E'] | parsed=B (explicit) | old extractor=B
   ....  Therefore, the most appropriate immediate next step is needle thoracostomy at the 2nd intercostal space, midclavicular line.  The final answer is $\boxed{B}$

[legacy_mcq_gemini_flash] item 48 | gold=B esc=['B', 'E'] | parsed=B (explicit) | old extractor=B
   ...ated.  Therefore, the 

In [15]:
# download and removing the name mcq from files

import os, re, glob, shutil

PARSED_DIR = globals().get("PARSED_DIR", "/content/frontier_v2/parsed")
ZIP_BASE = "/content/frontier_v2_parsed"                     # -> /content/frontier_v2_parsed.zip

def strip_mcq(name):
    stem, ext = os.path.splitext(name)
    stem = re.sub(r"mcq", "", stem, flags=re.I)              # remove the word
    stem = re.sub(r"[_\-\s]{2,}", "_", stem).strip("_- ")    # tidy leftover separators: a__b -> a_b
    return stem + ext

renames = {}
for path in sorted(glob.glob(os.path.join(PARSED_DIR, "*"))):
    old = os.path.basename(path)
    new = strip_mcq(old)
    if new != old:
        assert not os.path.exists(os.path.join(PARSED_DIR, new)) and new not in renames.values(), f"{old} -> {new} would overwrite another file"
        renames[old] = new
for old, new in renames.items():
    os.rename(os.path.join(PARSED_DIR, old), os.path.join(PARSED_DIR, new))
    print(f"renamed  {old}  ->  {new}")
print(f"{len(renames)} file(s) renamed" if renames else "No file names contained 'mcq'")

zip_path = shutil.make_archive(ZIP_BASE, "zip", PARSED_DIR)   # parsed files at the top level of the zip
print("\nZipped:", sorted(os.listdir(PARSED_DIR)))
print("Stored at:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass

renamed  legacy_mcq_deepseek_parsed.jsonl  ->  legacy_deepseek_parsed.jsonl
renamed  legacy_mcq_gemini_flash_parsed.jsonl  ->  legacy_gemini_flash_parsed.jsonl
renamed  legacy_mcq_gpt_baseline_parsed.jsonl  ->  legacy_gpt_baseline_parsed.jsonl
renamed  legacy_mcq_llama3_70b_parsed.jsonl  ->  legacy_llama3_70b_parsed.jsonl
renamed  legacy_mcq_mistral_small_parsed.jsonl  ->  legacy_mistral_small_parsed.jsonl
renamed  legacy_mcq_qwen_parsed.jsonl  ->  legacy_qwen_parsed.jsonl
6 file(s) renamed

Zipped: ['legacy_deepseek_parsed.jsonl', 'legacy_gemini_flash_parsed.jsonl', 'legacy_gpt_baseline_parsed.jsonl', 'legacy_llama3_70b_parsed.jsonl', 'legacy_mismatches.csv', 'legacy_mistral_small_parsed.jsonl', 'legacy_qwen_parsed.jsonl', 'legacy_summary.csv', 'legacy_unparseable_rows.csv']
Stored at: /content/frontier_v2_parsed.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>